In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import statsmodels.api as sm
import scipy.stats as stats

import pandas as pd
from collections import Counter
import numpy as np

import matplotlib.pyplot as plt

from compcats.datasets.fmri import nsd
from compcats.utils import model_utils, seed_utils
from compcats.datasets.fmri.utils import nsd_ds_utils

import main_analysis as ma
import paths

# Set seed
SEED = 42
seed_utils.seed_everything(SEED)

# Directories
RESULTS_DIR = 'results/interpreting/outliers/nsd'
os.makedirs(RESULTS_DIR, exist_ok=True)
PLOTS_DIR = f'{RESULTS_DIR}/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)
# Paths
log_file_path = os.path.join(RESULTS_DIR, 'nsd_outliers.log')
nsd_results_df_name = 'nsd_outliers.csv'

# Variables
batch_size = 2048
frois = ['FFA', 'PPA', 'EBA']
subjects = [1, 2, 5, 7]
localizer = 'grillspector'
tthreshold = 7
model_name = 'resnet50imagenet1kv2'

# COCO
coco_animal_labels = ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe']
coco_face_labels = coco_animal_labels + ['person']
coco_train = COCO(paths.COCO_TRAIN_ANNOTATIONS_PATH)
coco_val = COCO(paths.COCO_VAL_ANNOTATIONS_PATH)

# NSD IDs
stim1000_nsdids = nsd_ds_utils.get_subset_nsdids(paths.NSD_DIR, 1000)

# Outliers
# Parameters
standardized_residual_threshold = 2.58

# All Stimuli
allstim, _ = nsd.get_subset_stimuli(paths.NSD_DIR)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
loading annotations into memory...
Done (t=7.92s)
creating index...
index created!
loading annotations into memory...
Done (t=0.37s)
creating index...
index created!


In [2]:
# Util functions
def get_image_labels(image_id, coco_obj):
    """
    Get the labels for an image given its ID from the COCO dataset.
    
    Args:
        image_id (int): The ID of the image.
        
    Returns:
        list: A list of labels (category names) associated with the image.
    """
    # Get all annotations for the given image
    annotation_ids = coco_obj.getAnnIds(imgIds=image_id)
    annotations = coco_obj.loadAnns(annotation_ids)
    
    # Get category IDs from annotations
    category_ids = {annotation['category_id'] for annotation in annotations}
    
    # Get category names from category IDs
    category_names = [coco_obj.loadCats(cat_id)[0]['name'] for cat_id in category_ids]
    
    return category_names

def add_image_labels_to_df(df):
    """
    Add labels to the DataFrame given the COCO object.
    
    Args:
        df (pd.DataFrame): The DataFrame to which to add the labels.
        
    Returns:
        pd.DataFrame: The DataFrame with the labels added.
    """
    # Create a copy of the dataframe to avoid the SettingWithCopyWarning
    df_copy = df.copy()
    
    # Create a function to get the appropriate COCO object based on the split
    def get_coco_obj(split):
        if split == 'train2017':
            return coco_train
        else:
            return coco_val
        
    # Apply the function to get labels for each row using the appropriate COCO object
    df_copy.loc[:, 'labels'] = df_copy.apply(
        lambda row: get_image_labels(row['cocoId'], get_coco_obj(row['cocoSplit'])), 
        axis=1
    )
    return df_copy

def min_max_normalize(array):
    """
    Normalize the input array using min-max normalization.

    Parameters:
    array (numpy.ndarray or torch.Tensor): The input array to normalize.

    Returns:
    numpy.ndarray or torch.Tensor: The normalized array.
    """
    min_val = array.min()
    max_val = array.max()
    normalized_array = (array - min_val) / (max_val - min_val)
    return normalized_array

def z_score_normalize(array):
    """
    Normalize the input array using z-score normalization.

    Parameters:
    array (numpy.ndarray or torch.Tensor): The input array to normalize.

    Returns:
    numpy.ndarray or torch.Tensor: The normalized array.
    """
    mean_val = array.mean()
    std_val = array.std()
    normalized_array = (array - mean_val) / std_val
    return normalized_array

# NSD: Subjectwise 9000 stimuli

In [ ]:
# Generate the csv with outliers

for froi in frois:
    froi_results_df_name = nsd_results_df_name.replace('.csv', f'_{froi}.csv')
    # Check if the results file already exists
    if os.path.exists(os.path.join(RESULTS_DIR, froi_results_df_name)):
        print(f"Results file {froi_results_df_name} already exists. Skipping...")
        continue
    
    # Initialize the results dataframe
    results_df = None

    model = ma.get_model(model_name)
    all_models_layer = ma.get_froi_models_layers(froi, froi)
    layer = all_models_layer[model_name]

    model_allnsdstim_activations = model_utils.get_dataset_activations(
        MODELS_DIR, paths.NSD_DIR, model, layer, dataset='nsd', froi=froi, tthreshold=tthreshold, \
        froi_localizer=localizer, froi_localizer_dir=ma.get_localizer_dir(localizer), \
        batch_size=batch_size, device='cpu'
    )

    for subject in subjects:
        #
        # Subject response
        #
        # Get subject data
        subject_data, subject_stim_map = nsd.get_betas(
            paths.NSD_DIR, subject, froi=froi, tthreshold=tthreshold, voxels=False, repetition_mean=True
        )
        # Filter out nsdIds that are in stim1000_nsdids
        subject_stim_map_9000 = subject_stim_map[~subject_stim_map['nsdId'].isin(stim1000_nsdids)]
        subject_data_9000 = subject_data[~subject_stim_map['nsdId'].isin(stim1000_nsdids)]
        # Add labels
        subject_stim_map_9000_labels = add_image_labels_to_df(subject_stim_map_9000)

        #
        # Model response
        #
        # Get model data
        model_subject_data_9000 = model_allnsdstim_activations.mean(axis=-1)[list(subject_stim_map_9000['nsdId'])]

        #
        # OLS
        #
        # Normalize the data
        normalized_model_data = z_score_normalize(model_subject_data_9000)
        normalized_subject_data = z_score_normalize(subject_data_9000)
        X = normalized_model_data.numpy()
        y = normalized_subject_data.numpy()
        # Outlier detection
        n = len(X)  # number of data points
        k = 1   # number of predictors
        # Degrees of freedom
        df = n - k - 1
        # Two-tailed p-value
        p_value = 2 * (1 - stats.t.cdf(standardized_residual_threshold, df))
        # print(f"P-value: {p_value:.4f}")
        # Fit OLS
        # Fit the model using statsmodels (now with intercept)
        X_with_intercept = sm.add_constant(X)
        ols_model = sm.OLS(y, X_with_intercept).fit()
        slope = ols_model.params[1].item()
        intercept = ols_model.params[0].item()
        # Add OLS summary to log file
        with open(log_file_path, 'a') as log_file:
            log_file.write(f"OLS summary for subject {subject}, froi {froi}:\n")
            log_file.write(ols_model.summary().as_text())
            log_file.write("\n\n")
        # print(ols_model.summary())
        standardized_residuals = ols_model.get_influence().resid_studentized_internal

        # Outlier parameters
        n = len(X)  # number of data points
        k = 1   # number of predictors
        # Degrees of freedom
        df = n - k - 1
        # Two-tailed p-value
        p_value = 2 * (1 - stats.t.cdf(standardized_residual_threshold, df))
        # print(f"P-value: {p_value:.4f}")

        #
        # Save the results
        #
        # Create the subject results dataframe
        subject_results_df = subject_stim_map_9000_labels.copy()
        # Drop the Unnamed: 0 column which is just an index
        subject_results_df = subject_results_df.drop(columns=['Unnamed: 0'])
        # Add froi
        subject_results_df['froi'] = froi
        # Add threshold
        subject_results_df['tthreshold'] = tthreshold
        # Add localizer
        subject_results_df['model_localizer'] = localizer
        # Add subject number
        subject_results_df['subject'] = subject
        # Add model name
        subject_results_df['model_name'] = model_name
        # Add model layer
        subject_results_df['model_layer'] = layer
        # Add the model and subject data
        subject_results_df['model_response'] = model_subject_data_9000
        subject_results_df['subject_response'] = subject_data_9000
        # Add the normalized data
        subject_results_df['model_response_zscored'] = normalized_model_data
        subject_results_df['subject_response_zscored'] = normalized_subject_data
        # Add OLS details
        subject_results_df['ols_slope'] = slope
        subject_results_df['ols_intercept'] = intercept
        subject_results_df['ols_residuals'] = ols_model.resid
        subject_results_df['ols_standardized_residuals'] = standardized_residuals
        # Add predicted values from the OLS model
        subject_results_df['ols_predicted_values'] = ols_model.predict()
        # Add the R-squared value
        subject_results_df['ols_r_squared'] = ols_model.rsquared
        # Add the pearson correlation coefficient
        subject_results_df['ols_pearson_correlation'] = ols_model.rsquared ** 0.5
        assert np.isclose(ols_model.rsquared ** 0.5, np.corrcoef(X.flatten(), y.flatten())[0, 1], atol=1e-5)
        # Add the p-value for the slope
        subject_results_df['ols_slope_pvalue'] = ols_model.pvalues[1]
        # Add the p-value for the intercept
        subject_results_df['ols_intercept_pvalue'] = ols_model.pvalues[0]
        # Add the p-value for the R-squared value
        subject_results_df['ols_r_squared_pvalue'] = ols_model.f_pvalue
        # Add the p-value for the pearson correlation coefficient
        subject_results_df['ols_pearson_correlation_pvalue'] = ols_model.f_pvalue
        # Outliers
        # Add standardized residual outlier threshold
        subject_results_df['ols_standardized_residuals_threshold_tvalue'] = standardized_residual_threshold
        # Add the p-value
        subject_results_df['ols_standardized_residuals_threshold_pvalue'] = p_value
        # Add boolean flag for outliers (standardized residuals above or below standardized_residual_threshold)
        subject_results_df['is_outlier'] = abs(standardized_residuals) > standardized_residual_threshold
        # Add if the outlier is positive or negative
        subject_results_df['outlier_subject>model'] = standardized_residuals > standardized_residual_threshold
        subject_results_df['outlier_model>subject'] = standardized_residuals < -standardized_residual_threshold

        # Join the subject results to overall results df
        if results_df is None:
            results_df = subject_results_df
        else:
            # Concatenate the dataframes
            results_df = pd.concat([results_df, subject_results_df], ignore_index=True)
    # Save the results
    results_df.to_csv(
        os.path.join(RESULTS_DIR, froi_results_df_name),
        index=False
    )

In [35]:
# Plotting outliers

# froi = 'FFA'
for froi in frois:
    plot_dir = os.path.join(PLOTS_DIR, froi)
    os.makedirs(plot_dir, exist_ok=True)
    froi_results_df_name = nsd_results_df_name.replace('.csv', f'_{froi}.csv')
    # Load the results
    results_df = pd.read_csv(os.path.join(RESULTS_DIR, froi_results_df_name))
    # Get unique subjects
    subjects_in_df = results_df['subject'].unique()

    # Create a figure with 4 subplots in a single row
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=300)

    # Loop through each subject and create a subplot
    for i, subject in enumerate(subjects_in_df):
        # Filter data for the current subject
        subject_data = results_df[results_df['subject'] == subject]
        
        # Get model and subject responses
        X = subject_data['model_response_zscored']
        y = subject_data['subject_response_zscored']
        
        # Calculate R-squared value (get from first row since it's the same for all rows of this subject)
        pearson_correlation = subject_data['ols_pearson_correlation'].iloc[0]
        
        # Plot the data
        # Plot the points which are not outliers with gray markers alpha = 0.5
        # Plot the points which are outliers as gray markers with alpha = 0.5 and black border
        axes[i].scatter(X[~subject_data['is_outlier']], y[~subject_data['is_outlier']], color='gray', alpha=0.5)
        axes[i].scatter(X[subject_data['is_outlier']], y[subject_data['is_outlier']], color='gray', alpha=0.8, edgecolor='black')
        # axes[i].scatter(X, y, color='gray', alpha=0.5)
        axes[i].set_xlabel('Model response (z-scored)')
        axes[i].set_ylabel('Subject response (z-scored)')
        axes[i].set_title(rf'Subject {subject} - pearson $\rho$ = {pearson_correlation:.2f}')
        
        # Set equal aspect ratio
        axes[i].set_aspect('equal')
        # Get limits for this subplot
        max_limit = max(X.max(), y.max())
        min_limit = min(X.min(), y.min())
        pad = 0.05 * (max_limit - min_limit)
        max_limit += pad
        min_limit -= pad
        # Set limits
        axes[i].set_xlim(min_limit, max_limit)
        axes[i].set_ylim(min_limit, max_limit)
        
        # Add the regression line by loading slope and intercept from the first row
        slope = subject_data['ols_slope'].iloc[0]
        intercept = subject_data['ols_intercept'].iloc[0]
        x_line = np.linspace(min_limit, max_limit, 100)
        y_pred = (slope * x_line) + intercept
        axes[i].plot(x_line, y_pred, color='k', linestyle='-', alpha=0.8)
        # Get the standardized residual threshold from the first row and plot the lines for outlier detection
        standardized_residual_threshold = subject_data['ols_standardized_residuals_threshold_tvalue'].iloc[0]
        # Calculate prediction bounds based on standardized residuals
        residual_std = np.std(y - (slope * X - intercept))
        upper_bound = standardized_residual_threshold * residual_std
        lower_bound = -standardized_residual_threshold * residual_std
        # The outlier lines are around the regression line
        y_upper = (slope * x_line) + intercept + upper_bound
        y_lower = (slope * x_line) + intercept + lower_bound
        axes[i].plot(x_line, y_upper, color='k', linestyle='--', alpha=0.5)
        axes[i].plot(x_line, y_lower, color='k', linestyle='--', alpha=0.5)
        
        # Add grid
        # axes[i].grid(alpha=0.3)

        # Remove top and right spines
        axes[i].spines['top'].set_visible(False)
        axes[i].spines['right'].set_visible(False)

    plt.suptitle(f'{froi} - Model vs Subject Responses', fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, f'{froi}_nsd_outliers.png'), bbox_inches='tight')
    # plt.show()
    plt.close()

In [36]:
# Plotting overrepresentation

# froi = 'FFA'
for froi in frois:
    plot_dir = os.path.join(PLOTS_DIR, froi)
    os.makedirs(plot_dir, exist_ok=True)
    froi_results_df_name = nsd_results_df_name.replace('.csv', f'_{froi}.csv')
    # Load the results
    results_df = pd.read_csv(os.path.join(RESULTS_DIR, froi_results_df_name))

    # Get all labels
    all_labels = []
    for labels in results_df['labels']:
        all_labels.extend(eval(labels))
    # Count occurrences of each label
    label_counts = Counter(all_labels)
    # Sort labels by count
    sorted_label_counts = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)

    # Get outliers with human > model
    outliers_human_gt_model = results_df[results_df['outlier_subject>model'] == True]
    # Count occurrences of each label in the human > model outliers
    outliers_human_gt_model_labels = []
    for labels in outliers_human_gt_model['labels']:
        outliers_human_gt_model_labels.extend(eval(labels))
    # Count occurrences of each label
    outliers_human_gt_model_label_counts = Counter(outliers_human_gt_model_labels)
    # Sort labels by count
    outliers_human_gt_model_sorted_label_counts = sorted(outliers_human_gt_model_label_counts.items(), key=lambda x: x[1], reverse=True)

    # Get outliers with model > human
    outliers_model_gt_human = results_df[results_df['outlier_model>subject'] == True]
    # Count occurrences of each label in the model > human outliers
    outliers_model_gt_human_labels = []
    for labels in outliers_model_gt_human['labels']:
        outliers_model_gt_human_labels.extend(eval(labels))
    # Count occurrences of each label
    outliers_model_gt_human_label_counts = Counter(outliers_model_gt_human_labels)
    # Sort labels by count
    outliers_model_gt_human_sorted_label_counts = sorted(outliers_model_gt_human_label_counts.items(), key=lambda x: x[1], reverse=True)

    # Calculate the overrepresentation of labels in human > model outliers
    overrepresentation_human_gt_model = {}
    for label, count in outliers_human_gt_model_sorted_label_counts:
        if label in label_counts:
            overrepresentation_human_gt_model[label] = count / label_counts[label]
        else:
            raise ValueError(f"Label {label} not found in label counts.")
    # Sort by overrepresentation
    overrepresentation_human_gt_model_sorted = sorted(overrepresentation_human_gt_model.items(), key=lambda x: x[1], reverse=True)

    # Calculate the overrepresentation of labels in model > human outliers
    overrepresentation_model_gt_human = {}
    for label, count in outliers_model_gt_human_sorted_label_counts:
        if label in label_counts:
            overrepresentation_model_gt_human[label] = count / label_counts[label]
        else:
            raise ValueError(f"Label {label} not found in label counts.")
    # Sort by overrepresentation
    overrepresentation_model_gt_human_sorted = sorted(overrepresentation_model_gt_human.items(), key=lambda x: x[1], reverse=True)

    # Plot the top 25 overrepresented labels for human > model outliers
    top_n = 25
    top_human_gt_model = overrepresentation_human_gt_model_sorted[:top_n]
    top_human_gt_model_labels = [label for label, _ in top_human_gt_model]
    top_human_gt_model_values = [value for _, value in top_human_gt_model]
    # Plotting
    plt.figure(figsize=(12, 10), dpi=300)
    plt.bar(range(len(top_human_gt_model_labels)), top_human_gt_model_values, color='gray')
    # Get the counts for each label
    counts = [outliers_human_gt_model_label_counts[label] for label in top_human_gt_model_labels]
    # Add value annotations above bars
    for i, (value, count) in enumerate(zip(top_human_gt_model_values, counts)):
        plt.text(i, value, f'n={count}', ha='center', va='bottom', rotation=0, fontsize=8)
    plt.xticks(range(len(top_human_gt_model_labels)), top_human_gt_model_labels, rotation=45)
    plt.ylabel('Proportion of occurrences that are outliers')
    plt.xlabel('Object category')
    plt.title(f'Top {top_n} Overrepresented Categories in Human > Model Outliers ({froi})')
    plt.tight_layout()
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.savefig(os.path.join(plot_dir, f'{froi}_human_gt_model_overrepresentation.png'), bbox_inches='tight')
    # plt.show()
    plt.close()

    # Plot the top 25 overrepresented labels for model > human outliers
    top_model_gt_human = overrepresentation_model_gt_human_sorted[:top_n]
    top_model_gt_human_labels = [label for label, _ in top_model_gt_human]
    top_model_gt_human_values = [value for _, value in top_model_gt_human]
    # Plotting
    plt.figure(figsize=(12, 10), dpi=300)
    plt.bar(range(len(top_model_gt_human_labels)), top_model_gt_human_values, color='gray')
    # Get the counts for each label
    counts = [outliers_model_gt_human_label_counts[label] for label in top_model_gt_human_labels]
    # Add value annotations above bars
    for i, (value, count) in enumerate(zip(top_model_gt_human_values, counts)):
        plt.text(i, value, f'n={count}', ha='center', va='bottom', rotation=0, fontsize=8)
    plt.xticks(range(len(top_model_gt_human_labels)), top_model_gt_human_labels, rotation=45)
    plt.ylabel('Proportion of occurrences that are outliers')
    plt.xlabel('Object category')
    plt.title(f'Top {top_n} Overrepresented Categories in Model > Human Outliers ({froi})')
    plt.tight_layout()
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.savefig(os.path.join(plot_dir, f'{froi}_model_gt_human_overrepresentation.png'), bbox_inches='tight')
    # plt.show()
    plt.close()

In [38]:
# Plotting images with highest standardized residuals, separately for human > model & model > human outliers

# froi = 'FFA'
for froi in frois:
    plot_dir = os.path.join(PLOTS_DIR, froi)
    os.makedirs(plot_dir, exist_ok=True)
    froi_results_df_name = nsd_results_df_name.replace('.csv', f'_{froi}.csv')
    # Load the results
    results_df = pd.read_csv(os.path.join(RESULTS_DIR, froi_results_df_name))

    # Get outliers with human > model
    outliers_human_gt_model = results_df[results_df['outlier_subject>model'] == True]
    # Sort by absolute value of standardized residuals
    outliers_human_gt_model = outliers_human_gt_model.sort_values(by='ols_standardized_residuals', ascending=False)
    # Get the top 25 outliers
    top_n = 25
    top_human_gt_model = outliers_human_gt_model.head(top_n)
    # Get the image IDs
    top_human_gt_model_image_ids = top_human_gt_model['nsdId'].tolist()
    # Get the images from all stimuli
    top_human_gt_model_images = allstim[top_human_gt_model_image_ids]
    # Get the labels
    top_human_gt_model_labels = top_human_gt_model['labels'].tolist()
    # Get the standardized residuals
    top_human_gt_model_standardized_residuals = top_human_gt_model['ols_standardized_residuals'].tolist()
    # Get the model responses
    top_human_gt_model_model_responses = top_human_gt_model['model_response_zscored'].tolist()
    # Get the subject responses
    top_human_gt_model_subject_responses = top_human_gt_model['subject_response_zscored'].tolist()
    # Plot the images in a 5*5 grid
    n_cols = 5
    n_rows = 5
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 15), dpi=300)
    # Loop through each image and create a subplot
    for i, (image, label, standardized_residual, model_response, subject_response) in enumerate(zip(top_human_gt_model_images, top_human_gt_model_labels, top_human_gt_model_standardized_residuals, top_human_gt_model_model_responses, top_human_gt_model_subject_responses)):
        ax = axes[i // n_cols, i % n_cols]
        ax.imshow(image)
        ax.axis('off')
        # Add the label
        ax.set_title(f'{label} - {standardized_residual:.2f}\nModel: {model_response:.2f} - Subject: {subject_response:.2f}', fontsize=8)
        # Set equal aspect ratio
        ax.set_aspect('equal')
    # Remove top and right spines
    # Adjust layout
    plt.tight_layout()
    plt.suptitle(f'{froi} - Top {top_n} Human > Model Outliers', fontsize=16)
    plt.savefig(os.path.join(plot_dir, f'{froi}_top_human_gt_model_outliers.png'), bbox_inches='tight')
    # plt.show()
    plt.close()

    # Get outliers with model > human
    outliers_model_gt_human = results_df[results_df['outlier_model>subject'] == True]
    # Sort by absolute value of standardized residuals
    outliers_model_gt_human = outliers_model_gt_human.sort_values(by='ols_standardized_residuals', ascending=True)
    # Get the top 25 outliers
    top_model_gt_human = outliers_model_gt_human.head(top_n)
    # Get the image IDs
    top_model_gt_human_image_ids = top_model_gt_human['nsdId'].tolist()
    # Get the images from all stimuli
    top_model_gt_human_images = allstim[top_model_gt_human_image_ids]
    # Get the labels
    top_model_gt_human_labels = top_model_gt_human['labels'].tolist()
    # Get the standardized residuals
    top_model_gt_human_standardized_residuals = top_model_gt_human['ols_standardized_residuals'].tolist()
    # Get the model responses
    top_model_gt_human_model_responses = top_model_gt_human['model_response_zscored'].tolist()
    # Get the subject responses
    top_model_gt_human_subject_responses = top_model_gt_human['subject_response_zscored'].tolist()
    # Plot the images in a 5*5 grid
    n_cols = 5
    n_rows = 5
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 15), dpi=300)
    # Loop through each image and create a subplot
    for i, (image, label, standardized_residual, model_response, subject_response) in enumerate(zip(top_model_gt_human_images, top_model_gt_human_labels, top_model_gt_human_standardized_residuals, top_model_gt_human_model_responses, top_model_gt_human_subject_responses)):
        ax = axes[i // n_cols, i % n_cols]
        ax.imshow(image)
        ax.axis('off')
        # Add the label
        ax.set_title(f'{label} - {standardized_residual:.2f}\nModel: {model_response:.2f} - Subject: {subject_response:.2f}', fontsize=8)
        # Set equal aspect ratio
        ax.set_aspect('equal')
    # Adjust layout
    plt.tight_layout()
    plt.suptitle(f'{froi} - Top {top_n} Model > Human Outliers', fontsize=16)
    plt.savefig(os.path.join(plot_dir, f'{froi}_top_model_gt_human_outliers.png'), bbox_inches='tight')
    # plt.show()
    plt.close()